# Transcriptomic Distance to Control

In [ ]:
%reload_ext autoreload
%autoreload 0

## Load packages

In [ ]:
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Add sc-ops to path
sys.path.append('/projects/site/pred/ihb-g-deco/USERS/adaml9/sc-ops')
sys.path.append(str(base_proj_dir))

from sc_ops.settings import *
from sc_ops.preprocessing._adata_ops import *
from sc_ops.preprocessing._pert_ops import *
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

## Load settings and specify parameters

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)
tables_dir = Path(settings.ANALYSIS.tables_dir)

In [ ]:
sample_key = "sample"
celltype_key = "annotation"
perturbation_key = "perturbation"

## Load and clean data

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_filtered.h5ad")
adata.obs = adata.obs.drop(columns=["cell_type"]).rename(columns={"condition": "perturbation", "annotation": "cell_type"})

In [ ]:
# Get number of cells per perturbation and cell type
cell_counts = adata.obs.groupby(["perturbation", "cell_type"]).size().reset_index(name="cell_count")
print(cell_counts.head())

In [ ]:
# Compute perturbation effects
res = compute_perturbation_screen_results(
    adata,
    celltype_col="cell_type",
    perturbation_col="perturbation",
    sample_col="sample",
    control_label="Control",
    metric="euclidean",
    min_cells_per_group=20,
    min_celltypes=2,    
)

# Extract results
perturbation_effects_df = res["perturbation_effects"]
diff_df = res["difference_vectors"]
agg_expr_df = res["agg_expr"]

# Normalize effect magnitudes and sharedness scores within each cell type
for feature in ["perturbation_effect_magnitude", "perturbation_effect_sharedness"]:
    perturbation_effects_df[f"{feature}_norm"] = perturbation_effects_df.groupby("cell_type")[feature].transform(lambda x: (x - x.min()) / (x.max() - x.min()))

In [ ]:
# Get effect_matrix from perturbation_effects_df
effect_matrix = perturbation_effects_df.pivot(index="perturbation", columns="cell_type", values="perturbation_effect_magnitude").fillna(0).T

# Sort effect matrix ["Stem Cells", "TA Cells", "Cycling TA Cells", "Enterocytes", "Goblet Cells", "Paneth Cells", "EEC Progenitors", "EC Cells", "X Cells", "D Cells", "I/N Cells", "K Cells"]
celltype_order = ["Stem Cells", "TA Cells", "Cycling TA Cells", "Enterocytes", "Goblet Cells", "Paneth Cells", "EEC Progenitors", "EC Cells", "X Cells", "D Cells", "I/N Cells", "K Cells"]
effect_matrix = effect_matrix.loc[celltype_order]

In [ ]:
# Sort based on summed effect across cell types
effect_matrix = effect_matrix.loc[:, effect_matrix.sum().sort_values(ascending=False).index]

# Cap cmap at 100
effect_matrix = effect_matrix.clip(upper=100)

In [ ]:
with plt.rc_context({"axes.grid": False}):
    sns.clustermap(effect_matrix, cmap="Greys", figsize=(15, 4), method="ward", row_cluster=False, col_cluster=True)

In [ ]:
# Save the average expression DataFrame for downstream analysis
agg_expr_df.to_csv(tables_dir / "average_expression_per_group.csv", index=False)

# Export effect matrix as CSV
effect_matrix.to_csv(tables_dir / "perturbation_effect_matrix.csv")

# Export diff_df as CSV
diff_df.to_csv(tables_dir / "perturbation_diff_vectors.csv", index=False)

# Export perturbation_effects_df as CSV
perturbation_effects_df.to_csv(tables_dir / "perturbation_effects_summary.csv", index=False)

### Show transcriptomic response amplitude vs cell type specific effect

In [ ]:
res_df_plot = perturbation_effects_df.copy()
res_df_plot["color"] = res_df_plot["perturbation_effect_magnitude"].apply(lambda x: "#B0B0B0" if x < 40 else x)

In [ ]:
import matplotlib.pyplot as plt

celltypes = ["Enterocytes", "EEC Progenitors", "EC Cells", "X Cells", "D Cells", "I/N Cells", "K Cells"]

fig, axes = plt.subplots(1, len(celltypes), figsize=(24, 4), sharex=True, sharey=True)

for ax, ct in zip(axes, celltypes):
    sub = res_df_plot[res_df_plot["cell_type"] == ct]

    # split into grey vs colored
    is_grey = sub["color"] == "#B0B0B0" # adjust if needed

    # grey points (background)
    ax.scatter(
        sub.loc[is_grey, "perturbation_effect_sharedness_norm"],  # adjust if needed
        sub.loc[is_grey, "perturbation_effect_magnitude_norm"],  # adjust if needed
        c="lightgray",
        s=80,
        alpha=0.7,
    )

    # colored points (foreground)
    ax.scatter(
        sub.loc[~is_grey, "perturbation_effect_sharedness_norm"],  # adjust if needed
        sub.loc[~is_grey, "perturbation_effect_magnitude_norm"],  # adjust if needed
        c=sub.loc[~is_grey, "color"],
        s=100,
        edgecolor="black",
        linewidth=0.5,
    )

    # Label significant points
    for _, row in sub[~is_grey].iterrows():
        ax.text(
            row["perturbation_effect_sharedness_norm"] + 0.02,  # small offset to the right
            row["perturbation_effect_magnitude_norm"],
            row["perturbation"],
            fontsize=8,
            verticalalignment="center",
        )

    ax.set_title(ct)
    ax.grid(False)

# shared labels
axes[0].set_ylabel("Perturbation effect magnitude")
for ax in axes:
    ax.set_xlabel("Perturbation effect sharedness")

plt.tight_layout()
plt.show()